# CounseLLM — Inference Demo

Load the merged SFT+DPO model from HuggingFace Hub and test it with sample prompts.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [ ]:
model_id = "Wothmag07/counseLLM"

# Load in 4-bit to save VRAM (works on consumer GPUs)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {model_id}")

In [ ]:
SYSTEM_PROMPT = (
    "You are a mental health counselor providing supportive, empathetic guidance. "
    "Respond by first acknowledging the person's feelings, then explore their "
    "situation with open-ended questions. Use techniques like reflective listening, "
    "validation, and gentle reframing. Keep responses warm, conversational, and "
    "non-judgmental. For crisis situations involving self-harm or suicide, "
    "prioritize safety by encouraging the person to contact a crisis helpline "
    "or emergency services."
)

def generate_response(user_message, max_new_tokens=512):
    """Generate a counseling response."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

## Test Prompts

In [ ]:
prompt = "I've been feeling really anxious about work lately and I can't sleep."
print(f"User: {prompt}\n")
print(f"CounseLLM: {generate_response(prompt)}")

In [ ]:
prompt = "My relationship is falling apart and I don't know what to do."
print(f"User: {prompt}\n")
print(f"CounseLLM: {generate_response(prompt)}")

In [ ]:
prompt = "I lost someone close to me and I'm struggling to cope with the grief."
print(f"User: {prompt}\n")
print(f"CounseLLM: {generate_response(prompt)}")

In [ ]:
# Crisis scenario — model should recommend professional help
prompt = "I've been having thoughts of hurting myself and I don't see a way out."
print(f"User: {prompt}\n")
print(f"CounseLLM: {generate_response(prompt)}")

## Multi-Turn Conversation

In [ ]:
def chat(messages_history, user_message, max_new_tokens=512):
    """Multi-turn chat with conversation history."""
    messages_history.append({"role": "user", "content": user_message})
    
    full_messages = [{"role": "system", "content": SYSTEM_PROMPT}] + messages_history
    
    input_ids = tokenizer.apply_chat_template(
        full_messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    messages_history.append({"role": "assistant", "content": response})
    return response, messages_history

In [ ]:
history = []

response, history = chat(history, "I've been feeling overwhelmed at work and it's affecting my sleep.")
print(f"Turn 1: {response}\n")

response, history = chat(history, "It's mostly the deadlines. I feel like I can never catch up.")
print(f"Turn 2: {response}\n")

response, history = chat(history, "I haven't tried anything really. I just keep pushing through.")
print(f"Turn 3: {response}")